# 🎯 Sentiment Classification — End-to-End Notebook

**Mục tiêu:** Phân loại review là Tích cực (`1`) hoặc Tiêu cực (`0`)

**Metric:** F1-score (macro-averaged)

---

## 📋 Luồng thực hiện

```
1. Setup & Install
2. Load Data
3. EDA (Exploratory Data Analysis)
4. Text Preprocessing
5. Baseline: TF-IDF + Logistic Regression
6. Nâng cao: BERT Fine-tuning + 5-Fold CV
7. Evaluation & Error Analysis
8. Inference & Submission
```

---
## 1. 🔧 Setup & Install

In [ ]:
# Cài các thư viện cần thiết (chạy 1 lần trên Kaggle)
# Kaggle đã có sẵn: numpy, pandas, sklearn, matplotlib, seaborn
# Cần cài thêm: transformers, wordcloud, lightgbm

!pip install transformers datasets accelerate wordcloud lightgbm -q

In [ ]:
# ============================================================
# IMPORT TẤT CẢ THƯ VIỆN CẦN THIẾT
# ============================================================

# --- Cơ bản ---
import numpy as np
import pandas as pd
import re
import os
import warnings
warnings.filterwarnings('ignore')

# --- Visualization ---
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

# --- NLP ---
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
import scipy.sparse as sp

# --- Machine Learning ---
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from lightgbm import LGBMClassifier

# --- Deep Learning (BERT) ---
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW

# --- Reproducibility: đảm bảo kết quả giống nhau mỗi lần chạy ---
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# --- Kiểm tra GPU ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Sử dụng thiết bị: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

---
## 2. 📂 Load Data

In [ ]:
# ============================================================
# LOAD DATA
# ============================================================
# Thay đường dẫn theo dataset của bạn trên Kaggle
DATA_DIR = '/kaggle/input/datasets/namnguynnnn/sentence-classification/Dataset/'

train = pd.read_csv(DATA_DIR + 'train.csv')
test  = pd.read_csv(DATA_DIR + 'test.csv')

print('=' * 50)
print(f'Train shape: {train.shape}  →  {train.shape[0]} mẫu, {train.shape[1]} cột')
print(f'Test shape:  {test.shape}   →  {test.shape[0]} mẫu, {test.shape[1]} cột')
print('=' * 50)

# Xem cấu trúc dữ liệu
print('\n--- Vài dòng đầu của train ---')
display(train.head())

print('\n--- Vài dòng đầu của test ---')
display(test.head())

# Kiểm tra kiểu dữ liệu và missing values
print('\n--- Thông tin cột ---')
print(train.info())

print('\n--- Missing values trong train ---')
print(train.isnull().sum())

print('\n--- Missing values trong test ---')
print(test.isnull().sum())

In [ ]:
# Xử lý missing values nếu có
# fillna bằng chuỗi rỗng để tránh lỗi khi xử lý text
train['Text'] = train['Text'].fillna('')
test['Text']  = test['Text'].fillna('')

# Xóa dòng trùng lặp (nếu có)
n_before = len(train)
train = train.drop_duplicates(subset=['Text']).reset_index(drop=True)
print(f'Đã xóa {n_before - len(train)} dòng trùng lặp')
print(f'Train sau khi clean: {train.shape}')

---
## 3. 📊 EDA — Exploratory Data Analysis

> **Mục đích:** Hiểu data trước khi build model. Biết data có vấn đề gì để xử lý đúng cách.

In [ ]:
# ============================================================
# 3.1 PHÂN BỐ NHÃN (Label Distribution)
# ============================================================
# ❓ Tại sao quan trọng?
# Nếu data imbalanced (ví dụ 90% class 1), model sẽ "lười":
# predict toàn class 1 vẫn đạt accuracy 90% nhưng F1-macro thấp.
# Phải biết trước để xử lý!

label_counts = train['Label'].value_counts()
label_pct    = train['Label'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Biểu đồ cột
sns.countplot(x='Label', data=train, palette='Set2', ax=axes[0])
axes[0].set_title('Số lượng mẫu theo Label', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Label (0=Negative, 1=Positive)')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}',
                     (p.get_x() + p.get_width()/2, p.get_height()),
                     ha='center', va='bottom', fontsize=12)

# Biểu đồ tròn
label_pct.plot.pie(autopct='%1.1f%%', ax=axes[1],
                   labels=['Positive (1)', 'Negative (0)'],
                   colors=['#66c2a5','#fc8d62'], startangle=90)
axes[1].set_title('Tỉ lệ phần trăm Label', fontsize=13, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

print('Label 0 (Negative):', label_counts[0], f'({label_pct[0]:.1f}%)')
print('Label 1 (Positive):', label_counts[1], f'({label_pct[1]:.1f}%)')

# Đánh giá mức độ imbalanced
ratio = label_counts.max() / label_counts.min()
if ratio < 1.5:
    print(f'\n✅ Data khá balanced (ratio={ratio:.2f}) → Không cần xử lý đặc biệt')
elif ratio < 3:
    print(f'\n⚠️  Data hơi imbalanced (ratio={ratio:.2f}) → Nên dùng class_weight="balanced"')
else:
    print(f'\n❌ Data rất imbalanced (ratio={ratio:.2f}) → Cần oversampling hoặc class_weight')

In [ ]:
# ============================================================
# 3.2 PHÂN TÍCH ĐỘ DÀI VĂN BẢN
# ============================================================
# ❓ Tại sao?
# - Biết được MAX_LEN phù hợp cho BERT
# - char_len/word_count có thể là feature hữu ích cho ML model
# - Phân phối khác nhau giữa 2 class → gợi ý pattern

train['char_len']   = train['Text'].apply(len)
train['word_count'] = train['Text'].apply(lambda x: len(x.split()))
train['avg_word_len'] = train['Text'].apply(
    lambda x: np.mean([len(w) for w in x.split()]) if x.split() else 0
)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

for ax, col, title in zip(axes,
                           ['char_len', 'word_count', 'avg_word_len'],
                           ['Số ký tự', 'Số từ', 'Độ dài từ trung bình']):
    sns.histplot(data=train, x=col, hue='Label', bins=50,
                 palette='Set2', alpha=0.7, ax=ax)
    ax.set_title(f'{title} theo Label', fontsize=12, fontweight='bold')
    ax.set_xlabel(col)

plt.tight_layout()
plt.show()

# Thống kê mô tả theo từng label
print('--- Thống kê theo Label ---')
display(train.groupby('Label')[['char_len','word_count','avg_word_len']]
        .agg(['mean','median','max']).round(1))

# Gợi ý MAX_LEN cho BERT
p95 = int(np.percentile(train['word_count'], 95))
p99 = int(np.percentile(train['word_count'], 99))
print(f'\n📌 95th percentile word count: {p95} từ → dùng MAX_LEN={min(p95*2, 512)} token')
print(f'   99th percentile word count: {p99} từ')

In [ ]:
# ============================================================
# 3.3 WORDCLOUD — Từ nào xuất hiện nhiều ở mỗi class?
# ============================================================
# ❓ Tại sao?
# Giúp xác nhận data có ý nghĩa:
# Positive → xuất hiện: "great", "love", "excellent", "recommend"...
# Negative → xuất hiện: "bad", "terrible", "waste", "worst"...

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

for label, name, color, ax in [(1, 'Positive ✅', 'Greens', axes[0]),
                                 (0, 'Negative ❌', 'Reds',   axes[1])]:
    text = ' '.join(train[train['Label'] == label]['Text'])
    wc = WordCloud(
        width=800, height=400,
        background_color='white',
        colormap=color,
        max_words=100,
        random_state=SEED
    ).generate(text)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(f'WordCloud — {name}', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 3.4 TOP N-GRAM ANALYSIS
# ============================================================
# Xem top bigram (2-gram) đặc trưng cho từng class
# "not good", "very bad" → thường bị bỏ sót khi chỉ nhìn unigram

from sklearn.feature_extraction.text import CountVectorizer

def get_top_ngrams(texts, n=2, top_k=15):
    vec = CountVectorizer(ngram_range=(n,n), max_features=10000,
                          stop_words='english').fit(texts)
    bag = vec.transform(texts)
    sum_words = bag.sum(axis=0)
    words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
    return sorted(words_freq, key=lambda x: x[1], reverse=True)[:top_k]

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

for label, name, color, ax in [(1, 'Positive', '#66c2a5', axes[0]),
                                 (0, 'Negative', '#fc8d62', axes[1])]:
    texts = train[train['Label'] == label]['Text']
    top_ng = get_top_ngrams(texts, n=2, top_k=15)
    df_ng = pd.DataFrame(top_ng, columns=['bigram', 'count'])
    sns.barplot(data=df_ng, x='count', y='bigram', color=color, ax=ax)
    ax.set_title(f'Top 15 Bigrams — {name}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Số lần xuất hiện')

plt.tight_layout()
plt.show()

---
## 4. 🧹 Text Preprocessing

> **Nguyên tắc:** Clean text cho ML model (TF-IDF). Giữ nguyên text gốc cho BERT.

In [ ]:
# ============================================================
# HÀM CLEAN TEXT CHO TF-IDF / ML MODELS
# ============================================================
# KHÔNG dùng cho BERT vì BERT đã được train trên text gốc

STOPWORDS = set(stopwords.words('english'))
# Giữ lại các từ phủ định quan trọng: "not", "no", "never", "neither"
# vì chúng thay đổi hoàn toàn nghĩa của câu
KEEP_WORDS = {'not', 'no', 'never', 'neither', 'nor', 'hardly', 'barely'}
STOPWORDS -= KEEP_WORDS

def clean_text(text: str, remove_stopwords: bool = False) -> str:
    """
    Làm sạch text cho TF-IDF / ML models.
    
    Args:
        text: chuỗi văn bản gốc
        remove_stopwords: có xóa stopwords không
                          (thường KHÔNG xóa vì TF-IDF tự xử lý)
    Returns:
        text đã được làm sạch
    """
    if not isinstance(text, str):
        return ''
    
    # (1) Lowercase — "Good" và "good" là 1 từ
    text = text.lower()
    
    # (2) Bỏ URL — không mang thông tin sentiment
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    
    # (3) Bỏ HTML tags — phổ biến trong review từ web
    text = re.sub(r'<.*?>', '', text)
    
    # (4) Normalize negation: "don't" → "do not", "isn't" → "is not"
    # Quan trọng! TF-IDF sẽ bắt được "not" như một feature
    contractions = {
        "don't": "do not", "doesn't": "does not", "didn't": "did not",
        "won't": "will not", "wouldn't": "would not", "can't": "cannot",
        "couldn't": "could not", "shouldn't": "should not",
        "isn't": "is not", "aren't": "are not", "wasn't": "was not",
        "weren't": "were not", "hadn't": "had not", "hasn't": "has not",
        "haven't": "have not", "mustn't": "must not", "needn't": "need not",
    }
    for contr, full in contractions.items():
        text = text.replace(contr, full)
    
    # (5) Bỏ ký tự đặc biệt, giữ lại chữ cái và khoảng trắng
    text = re.sub(r'[^a-z\s]', ' ', text)
    
    # (6) Chuẩn hóa khoảng trắng
    text = re.sub(r'\s+', ' ', text).strip()
    
    # (7) Tùy chọn: xóa stopwords
    if remove_stopwords:
        text = ' '.join(w for w in text.split() if w not in STOPWORDS)
    
    return text

# Áp dụng
train['clean_text'] = train['Text'].apply(clean_text)
test['clean_text']  = test['Text'].apply(clean_text)

# Kiểm tra kết quả
print('Ví dụ text trước và sau clean:')
for i in range(3):
    print(f'\n[{i}] BEFORE: {train["Text"].iloc[i][:100]}')
    print(f'[{i}] AFTER:  {train["clean_text"].iloc[i][:100]}')

---
## 5. ⚡ Baseline: TF-IDF + ML Models

> Chạy nhanh để có baseline score. Nếu baseline tốt thì BERT sẽ càng tốt hơn.

In [ ]:
# ============================================================
# 5.1 TẠO TF-IDF FEATURES
# ============================================================
# Dùng 2 loại vectorizer rồi ghép lại:
# Word n-gram: bắt cụm từ có nghĩa ("not good", "highly recommend")
# Char n-gram: bắt tiền/hậu tố, chịu được lỗi chính tả

print('🔄 Đang tạo TF-IDF features...')

# Word-level: unigram đến trigram
vec_word = TfidfVectorizer(
    ngram_range=(1, 3),       # 1-gram, 2-gram, 3-gram
    max_features=100_000,     # giữ 100k features phổ biến nhất
    sublinear_tf=True,        # log(tf+1) thay vì tf thô → giảm ảnh hưởng từ lặp
    min_df=2,                 # bỏ từ chỉ xuất hiện 1 lần (noise)
    analyzer='word'
)

# Char-level: 2 đến 5 ký tự
vec_char = TfidfVectorizer(
    ngram_range=(2, 5),
    max_features=100_000,
    sublinear_tf=True,
    min_df=2,
    analyzer='char_wb'        # char_wb: xét trong từng từ, không across từ
)

# Fit trên train, transform cả train và test
X_train_w = vec_word.fit_transform(train['clean_text'])
X_test_w  = vec_word.transform(test['clean_text'])

X_train_c = vec_char.fit_transform(train['clean_text'])
X_test_c  = vec_char.transform(test['clean_text'])

# Ghép word + char features
X_train_tfidf = sp.hstack([X_train_w, X_train_c], format='csr')
X_test_tfidf  = sp.hstack([X_test_w, X_test_c], format='csr')

y_train = train['Label'].values

print(f'✅ Train TF-IDF shape: {X_train_tfidf.shape}')
print(f'   Test  TF-IDF shape: {X_test_tfidf.shape}')

In [ ]:
# ============================================================
# 5.2 CROSS-VALIDATION VỚI NHIỀU MODEL
# ============================================================
# StratifiedKFold: đảm bảo mỗi fold có tỉ lệ class giống nhau
# Tránh fold bị lệch class → đánh giá không chính xác

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

models = {
    'Logistic Regression': LogisticRegression(
        C=5,                    # C nhỏ = regularization mạnh hơn
        max_iter=1000,
        solver='saga',          # saga tốt cho sparse data lớn
        class_weight='balanced' # tự động điều chỉnh nếu imbalanced
    ),
    'LightGBM': LGBMClassifier(
        n_estimators=300,
        learning_rate=0.1,
        num_leaves=63,
        class_weight='balanced',
        n_jobs=-1,
        random_state=SEED,
        verbose=-1
    )
}

print('🔄 Đang chạy Cross-Validation...\n')
results = {}

for name, model in models.items():
    scores = cross_val_score(
        model, X_train_tfidf, y_train,
        cv=skf,
        scoring='f1_macro',
        n_jobs=-1
    )
    results[name] = scores
    print(f'{name}:')
    print(f'  Fold scores: {[f"{s:.4f}" for s in scores]}')
    print(f'  Mean F1: {scores.mean():.4f} ± {scores.std():.4f}\n')

# So sánh visually
fig, ax = plt.subplots(figsize=(8, 4))
ax.boxplot([results[m] for m in models], labels=list(models.keys()))
ax.set_title('F1-macro CV Scores by Model')
ax.set_ylabel('F1-macro')
ax.axhline(y=0.9, color='red', linestyle='--', alpha=0.5, label='0.9 threshold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 5.3 TRAIN BEST ML MODEL TRÊN TOÀN BỘ TRAIN SET
# ============================================================
# Sau khi chọn model tốt nhất qua CV, train lại trên 100% data

best_model_name = max(results, key=lambda k: results[k].mean())
best_model = models[best_model_name]

print(f'🏆 Best model: {best_model_name}')
print(f'   CV F1-macro: {results[best_model_name].mean():.4f}')
print('\n🔄 Training trên toàn bộ train set...')

best_model.fit(X_train_tfidf, y_train)
baseline_test_preds = best_model.predict(X_test_tfidf)

print('✅ Done! Baseline predictions sẵn sàng.')

---
## 6. 🧠 BERT Fine-tuning + 5-Fold CV

> BERT hiểu ngữ cảnh, nên cho kết quả tốt hơn nhiều so với TF-IDF.

In [ ]:
# ============================================================
# 6.1 CONFIG
# ============================================================
# Chọn model phù hợp với GPU của bạn:
# - distilbert-base-uncased: nhẹ nhất, nhanh nhất
# - bert-base-uncased: cân bằng
# - roberta-base: tốt nhất cho text classification

MODEL_NAME = 'distilbert-base-uncased'  # Đổi thành roberta-base nếu GPU mạnh
MAX_LEN    = 256   # Token tối đa (tùy độ dài review)
BATCH_SIZE = 16    # Giảm xuống 8 nếu OOM (Out of Memory)
EPOCHS     = 3     # 3 epoch thường đủ cho BERT fine-tuning
LR         = 2e-5  # Learning rate nhỏ — fine-tuning không train từ đầu
WARMUP_RATIO = 0.1 # 10% đầu tăng LR từ từ
N_FOLDS    = 5

print(f'Model:      {MODEL_NAME}')
print(f'MAX_LEN:    {MAX_LEN}')
print(f'Batch size: {BATCH_SIZE}')
print(f'Epochs:     {EPOCHS}')
print(f'LR:         {LR}')
print(f'Device:     {DEVICE}')

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'\n✅ Tokenizer loaded: {MODEL_NAME}')

In [ ]:
# ============================================================
# 6.2 DATASET CLASS
# ============================================================
# PyTorch Dataset: định nghĩa cách đọc từng mẫu dữ liệu

class ReviewDataset(Dataset):
    """
    Dataset cho sentiment classification.
    
    - texts:  list các chuỗi text (dùng text GỐC, không clean)
    - labels: list nhãn (None nếu là test set)
    """
    def __init__(self, texts, labels=None):
        self.texts  = list(texts)
        self.labels = list(labels) if labels is not None else None
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        # Tokenize: chuyển text → input_ids, attention_mask
        encoding = tokenizer(
            self.texts[idx],
            max_length=MAX_LEN,
            padding='max_length',   # pad ngắn hơn MAX_LEN bằng [PAD]
            truncation=True,        # cắt bớt nếu dài hơn MAX_LEN
            return_tensors='pt'     # trả về PyTorch tensor
        )
        
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        
        return item

print('✅ ReviewDataset class định nghĩa xong')

In [ ]:
# ============================================================
# 6.3 TRAINING FUNCTIONS
# ============================================================

def train_one_epoch(model, loader, optimizer, scheduler, epoch_num):
    """
    Train model 1 epoch.
    Trả về: average loss của epoch đó
    """
    model.train()
    total_loss = 0
    n_batches  = len(loader)
    
    for batch_idx, batch in enumerate(loader):
        # Chuyển data lên GPU
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        
        # Forward pass
        outputs = model(**batch)
        loss = outputs.loss  # CrossEntropyLoss tự tính trong model
        
        # Backward pass
        loss.backward()
        
        # Gradient clipping: tránh gradient quá lớn → training không ổn định
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        # Update weights
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        
        total_loss += loss.item()
        
        # In progress mỗi 20% batch
        if (batch_idx + 1) % max(1, n_batches // 5) == 0:
            print(f'    Batch {batch_idx+1}/{n_batches} | Loss: {loss.item():.4f}')
    
    return total_loss / n_batches


@torch.no_grad()
def evaluate(model, loader):
    """
    Đánh giá model trên validation set.
    Trả về: predictions (numpy array)
    """
    model.eval()  # tắt dropout, batch norm ở eval mode
    all_preds = []
    all_probs = []
    
    for batch in loader:
        # Bỏ labels khi predict (model không cần)
        inputs = {k: v.to(DEVICE) for k, v in batch.items() if k != 'labels'}
        outputs = model(**inputs)
        
        # Lấy logits → softmax → class có xác suất cao nhất
        probs = torch.softmax(outputs.logits, dim=-1)
        preds = outputs.logits.argmax(dim=-1)
        
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
    
    return np.array(all_preds), np.array(all_probs)


print('✅ Training functions định nghĩa xong')

In [ ]:
# ============================================================
# 6.4 5-FOLD CROSS-VALIDATION + OOF PREDICTIONS
# ============================================================
# OOF (Out-of-Fold): prediction cho mỗi sample khi nó là validation
# → Có thể tính CV score mà không bị data leakage
# → Có thể dùng để ensemble với các model khác

texts_all  = train['Text'].values   # Text GỐC cho BERT
labels_all = train['Label'].values
texts_test = test['Text'].values

# Lưu kết quả
oof_preds  = np.zeros(len(train), dtype=int)
oof_probs  = np.zeros((len(train), 2))  # xác suất 2 class
test_probs_list = []  # lưu probs của test set qua mỗi fold

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

for fold, (tr_idx, val_idx) in enumerate(skf.split(texts_all, labels_all)):
    print(f'\n{"="*60}')
    print(f'FOLD {fold+1}/{N_FOLDS}')
    print(f'  Train: {len(tr_idx)} mẫu | Val: {len(val_idx)} mẫu')
    print(f'{"="*60}')
    
    # --- Tạo DataLoader ---
    tr_dataset   = ReviewDataset(texts_all[tr_idx],  labels_all[tr_idx])
    val_dataset  = ReviewDataset(texts_all[val_idx], labels_all[val_idx])
    test_dataset = ReviewDataset(texts_test)
    
    tr_loader   = DataLoader(tr_dataset,   batch_size=BATCH_SIZE,   shuffle=True,  num_workers=2)
    val_loader  = DataLoader(val_dataset,  batch_size=BATCH_SIZE*2, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE*2, shuffle=False, num_workers=2)
    
    # --- Load model (mỗi fold load lại từ pretrained) ---
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2  # binary classification
    ).to(DEVICE)
    
    # --- Optimizer & Scheduler ---
    # AdamW: Adam với weight decay đúng cách (không decay bias)
    optimizer = AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=0.01  # L2 regularization
    )
    
    total_steps   = len(tr_loader) * EPOCHS
    warmup_steps  = int(total_steps * WARMUP_RATIO)
    
    # Linear warmup + linear decay
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )
    
    # --- Training Loop ---
    best_f1    = 0
    best_state = None
    
    for epoch in range(EPOCHS):
        print(f'\n  Epoch {epoch+1}/{EPOCHS}')
        
        # Train
        avg_loss = train_one_epoch(model, tr_loader, optimizer, scheduler, epoch)
        
        # Validate
        val_preds, val_probs = evaluate(model, val_loader)
        f1 = f1_score(labels_all[val_idx], val_preds, average='macro')
        
        print(f'  → Train Loss: {avg_loss:.4f} | Val F1-macro: {f1:.4f}')
        
        # Lưu model tốt nhất (Early stopping theo F1)
        if f1 > best_f1:
            best_f1 = f1
            # Lưu state dict lên CPU để tiết kiệm VRAM
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            print(f'  ✅ New best F1: {best_f1:.4f} — model saved!')
    
    # --- Load best model và predict ---
    model.load_state_dict(best_state)
    model.to(DEVICE)
    
    # OOF predictions
    oof_preds[val_idx], oof_probs[val_idx] = evaluate(model, val_loader)
    
    # Test predictions
    _, test_probs = evaluate(model, test_loader)
    test_probs_list.append(test_probs)
    
    fold_f1 = f1_score(labels_all[val_idx], oof_preds[val_idx], average='macro')
    print(f'\n  🏆 Fold {fold+1} Best F1: {fold_f1:.4f}')
    
    # Giải phóng VRAM
    del model
    torch.cuda.empty_cache()

# --- Tổng kết OOF ---
oof_f1 = f1_score(labels_all, oof_preds, average='macro')
print(f'\n{"="*60}')
print(f'🎯 OOF F1-macro (BERT): {oof_f1:.4f}')
print(f'{"="*60}')

---
## 7. 📈 Evaluation & Error Analysis

In [ ]:
# ============================================================
# 7.1 CLASSIFICATION REPORT & CONFUSION MATRIX
# ============================================================

print('=== BERT OOF Evaluation ===')
print(classification_report(
    labels_all, oof_preds,
    target_names=['Negative (0)', 'Positive (1)']
))

# Confusion Matrix
cm = confusion_matrix(labels_all, oof_preds)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred Neg', 'Pred Pos'],
            yticklabels=['True Neg', 'True Pos'],
            ax=ax)
ax.set_title(f'OOF Confusion Matrix\nF1-macro: {oof_f1:.4f}',
             fontsize=13, fontweight='bold')
ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')
plt.tight_layout()
plt.show()

# Tính thêm metrics thủ công
tn, fp, fn, tp = cm.ravel()
print(f'True Negative:  {tn} (Negative đúng)')
print(f'False Positive: {fp} (Negative nhưng predict Positive)')
print(f'False Negative: {fn} (Positive nhưng predict Negative)')
print(f'True Positive:  {tp} (Positive đúng)')

In [ ]:
# ============================================================
# 7.2 ERROR ANALYSIS — Xem những mẫu dự đoán SAI
# ============================================================
# Đây là bước cực kỳ quan trọng để cải thiện model!
# Nhìn vào lỗi sai → hiểu model đang yếu ở đâu → fix

train_eval = train.copy()
train_eval['oof_pred']       = oof_preds
train_eval['prob_negative']  = oof_probs[:, 0]
train_eval['prob_positive']  = oof_probs[:, 1]
train_eval['confidence']     = oof_probs.max(axis=1)
train_eval['is_wrong']       = train_eval['Label'] != train_eval['oof_pred']

errors = train_eval[train_eval['is_wrong']].sort_values('confidence', ascending=False)

print(f'Tổng số lỗi: {len(errors)} / {len(train_eval)} ({len(errors)/len(train_eval)*100:.1f}%)')
print()

# Phân loại lỗi
fp_errors = errors[errors['oof_pred'] == 1]  # Negative bị predict là Positive
fn_errors = errors[errors['oof_pred'] == 0]  # Positive bị predict là Negative
print(f'False Positives (Neg→Pos): {len(fp_errors)}')
print(f'False Negatives (Pos→Neg): {len(fn_errors)}')

# Xem những lỗi tự tin nhất (confidence cao mà vẫn sai)
print('\n--- 5 Lỗi tự tin nhất (Confident Errors) ---')
print('(Model rất tự tin nhưng vẫn sai → patterns khó)')
for _, row in errors.head(5).iterrows():
    true_name = 'Positive' if row['Label'] == 1 else 'Negative'
    pred_name = 'Positive' if row['oof_pred'] == 1 else 'Negative'
    print(f'\nText: {row["Text"][:120]}...')
    print(f'True: {true_name} | Predicted: {pred_name} | Confidence: {row["confidence"]:.3f}')

In [ ]:
# ============================================================
# 7.3 CONFIDENCE DISTRIBUTION
# ============================================================
# Xem model có tự tin không?
# Confidence thấp → model không chắc → có thể cải thiện

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Đúng vs Sai
sns.histplot(
    data=train_eval, x='confidence', hue='is_wrong',
    bins=50, ax=axes[0],
    palette={False: '#66c2a5', True: '#fc8d62'}
)
axes[0].set_title('Confidence: Đúng vs Sai')
axes[0].set_xlabel('Confidence')

# Phân phối prob_positive theo label thực
sns.histplot(
    data=train_eval, x='prob_positive', hue='Label',
    bins=50, ax=axes[1],
    palette={0: '#fc8d62', 1: '#66c2a5'}
)
axes[1].set_title('P(Positive) phân theo True Label')
axes[1].set_xlabel('P(Positive)')
axes[1].axvline(x=0.5, color='black', linestyle='--', label='threshold=0.5')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 8. 📤 Inference & Submission

In [ ]:
# ============================================================
# 8.1 ENSEMBLE: BERT FOLDS
# ============================================================
# Avg probability của 5 folds → predict class có prob cao hơn
# Avg prob tốt hơn majority vote vì tận dụng được mức độ tự tin

# Stack: shape (N_test, 2, N_FOLDS)
test_probs_stack = np.stack(test_probs_list, axis=-1)  # (N_test, 2, 5)

# Average probability qua 5 folds
test_avg_probs = test_probs_stack.mean(axis=-1)  # (N_test, 2)

# Predict: class có average probability cao nhất
bert_final_preds = test_avg_probs.argmax(axis=1)

print('BERT Ensemble (5-fold avg):')
print(f'  Class 0 (Negative): {(bert_final_preds == 0).sum()} mẫu')
print(f'  Class 1 (Positive): {(bert_final_preds == 1).sum()} mẫu')

In [ ]:
# ============================================================
# 8.2 (TÙY CHỌN) ENSEMBLE BERT + BASELINE
# ============================================================
# Kết hợp BERT probability + ML probability
# Nếu 2 model học được pattern khác nhau → ensemble sẽ mạnh hơn

# Lấy probability từ baseline ML model
baseline_probs = best_model.predict_proba(X_test_tfidf)  # (N_test, 2)

# Weighted average: BERT mạnh hơn nên weight cao hơn
BERT_WEIGHT     = 0.75
BASELINE_WEIGHT = 0.25

ensemble_probs = (BERT_WEIGHT     * test_avg_probs +
                  BASELINE_WEIGHT * baseline_probs)

ensemble_preds = ensemble_probs.argmax(axis=1)

print(f'Ensemble BERT({BERT_WEIGHT}) + Baseline({BASELINE_WEIGHT}):')
print(f'  Class 0: {(ensemble_preds == 0).sum()}')
print(f'  Class 1: {(ensemble_preds == 1).sum()}')

# So sánh với chỉ BERT
agreement = (bert_final_preds == ensemble_preds).mean()
print(f'  Đồng thuận với BERT-only: {agreement*100:.1f}%')

In [ ]:
# ============================================================
# 8.3 TẠO FILE SUBMISSION
# ============================================================

# Chọn predictions tốt nhất
# Nếu OOF validation cho thấy ensemble tốt hơn → dùng ensemble
# Nếu BERT-only đã tốt → dùng bert_final_preds
final_predictions = bert_final_preds  # Thay bằng ensemble_preds nếu muốn

submission = pd.DataFrame({
    'ID':    range(1, len(test) + 1),
    'Label': final_predictions
})

submission.to_csv('submission.csv', index=False)

print('✅ submission.csv đã tạo xong!')
print(f'   Tổng: {len(submission)} dòng')
print()
display(submission.head(10))

print('\nPhân phối predictions:')
print(submission['Label'].value_counts())
print(submission['Label'].value_counts(normalize=True).round(3))

In [ ]:
# ============================================================
# 8.4 SANITY CHECK TRƯỚC KHI SUBMIT
# ============================================================

sub_check = pd.read_csv('submission.csv')

checks = [
    ('Số dòng đúng', len(sub_check) == len(test)),
    ('Có cột ID',    'ID' in sub_check.columns),
    ('Có cột Label', 'Label' in sub_check.columns),
    ('Label chỉ là 0/1', set(sub_check['Label'].unique()).issubset({0, 1})),
    ('Không có NaN',  sub_check.isnull().sum().sum() == 0),
    ('ID từ 1 đến N', sub_check['ID'].min() == 1 and sub_check['ID'].max() == len(test)),
]

print('=== SANITY CHECK ===')
all_pass = True
for check_name, result in checks:
    status = '✅' if result else '❌'
    print(f'{status} {check_name}')
    if not result:
        all_pass = False

print()
if all_pass:
    print('🚀 Tất cả OK! Sẵn sàng submit!')
else:
    print('⚠️  Có lỗi! Kiểm tra lại trước khi submit.')

---
## 9. 🚀 Các bước cải thiện thêm (nếu muốn push F1 cao hơn)

| Kỹ thuật | Cách làm | Tác động |
|---|---|---|
| Dùng `roberta-base` | Đổi MODEL_NAME | +1–2% |
| Tăng MAX_LEN lên 512 | Cần GPU nhiều VRAM hơn | +0.5% |
| Pseudo-labeling | Train thêm với confident test preds | +0.5–1% |
| Text Augmentation | Back-translation, synonym replacement | +0.5% |
| Threshold tuning | Tối ưu threshold thay vì cố định 0.5 | +0.3% |
| Stochastic Weight Averaging | `torch.optim.swa_utils` | +0.3% |

In [ ]:
# ============================================================
# BONUS: TỐI ƯU THRESHOLD
# ============================================================
# Thay vì luôn dùng 0.5, tìm threshold tối ưu F1 trên OOF

from sklearn.metrics import f1_score

best_threshold = 0.5
best_threshold_f1 = 0

for threshold in np.arange(0.3, 0.7, 0.01):
    preds = (oof_probs[:, 1] >= threshold).astype(int)
    f1 = f1_score(labels_all, preds, average='macro')
    if f1 > best_threshold_f1:
        best_threshold_f1 = f1
        best_threshold = threshold

print(f'Default threshold (0.5): F1 = {oof_f1:.4f}')
print(f'Optimal threshold ({best_threshold:.2f}): F1 = {best_threshold_f1:.4f}')
print(f'Cải thiện: +{(best_threshold_f1 - oof_f1)*100:.2f}%')

# Áp dụng threshold tối ưu cho test predictions
final_predictions_tuned = (test_avg_probs[:, 1] >= best_threshold).astype(int)

submission_tuned = pd.DataFrame({
    'ID':    range(1, len(test) + 1),
    'Label': final_predictions_tuned
})
submission_tuned.to_csv('submission_threshold_tuned.csv', index=False)
print('\n✅ submission_threshold_tuned.csv đã tạo!')